In [1]:
import json, numpy as np, os, zipfile
from onnx import helper, save, TensorProto

DATA_DIR = '/kaggle/input/competitions/neurogolf-2026'
OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

BATCH, CH, H, W = 1, 10, 30, 30
DT = TensorProto.FLOAT

def mk_tensor(name, dtype, shape, data):
    return helper.make_tensor(name, dtype, shape, data)

def make_model(nodes, initializers):
    inp = helper.make_tensor_value_info('input', DT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info('output', DT, [BATCH, CH, H, W])
    graph = helper.make_graph(nodes, 'g', [inp], [out], initializers)
    return helper.make_model(graph, ir_version=11, opset_imports=[helper.make_opsetid('', 11)])

# 1x1 conv for color remapping
def make_1x1(mapping):
    W_arr = np.zeros((CH, CH, 1, 1), dtype=np.float32)
    B_arr = np.full((CH,), -0.5, dtype=np.float32)
    for ic, oc in mapping.items():
        W_arr[oc, ic, 0, 0] = 1.0
    nodes = [helper.make_node('Conv', ['input', 'W', 'B'], ['output'], kernel_shape=[1, 1])]
    return make_model(nodes, [mk_tensor('W', DT, [CH, CH, 1, 1], W_arr.flatten()), mk_tensor('B', DT, [CH], B_arr)])

In [2]:
# Analyze each task from training examples
for t in range(1, 401):
    fname = f'{DATA_DIR}/task{t:03d}.json'
    if not os.path.exists(fname): continue
    with open(fname) as f:
        d = json.load(f)
    train = d.get('train', [])
    if not train: continue
    
    ex0 = train[0]
    inp = np.array(ex0['input'])
    out = np.array(ex0['output'])
    IH, IW = inp.shape
    OH, OW = out.shape
    
    all_ex = train + d.get('test', [])
    
    ex_mappings = []
    for ex in all_ex:
        inp_i = np.array(ex['input'])
        out_i = np.array(ex['output'])
        if inp_i.shape != out_i.shape: continue
        m = {}
        for r in range(inp_i.shape[0]):
            for c in range(inp_i.shape[1]):
                ic, oc = int(inp_i[r,c]), int(out_i[r,c])
                if ic != oc:
                    m[ic] = oc
        ex_mappings.append(m)
    
    if ex_mappings:
        common = {}
        for k, v in ex_mappings[0].items():
            if all(em.get(k) == v for em in ex_mappings[1:]):
                common[k] = v
        print(f'T{t:02d} {IH}x{IW}->{OH}x{OW}: mapping={common}')
    else:
        print(f'T{t:02d} {IH}x{IW}->{OH}x{OW}: size change (identity)')

T01 3x3->9x9: size change (identity)
T02 6x6->6x6: mapping={0: 4}
T03 6x3->9x3: size change (identity)
T04 14x9->14x9: mapping={}
T05 21x21->21x21: mapping={}
T06 3x7->3x3: size change (identity)
T07 7x7->7x7: mapping={}
T08 14x9->14x9: mapping={2: 0, 0: 2}
T09 20x20->20x20: mapping={}
T10 9x9->9x9: mapping={}
T11 11x11->11x11: mapping={6: 0, 3: 0}
T12 12x12->12x12: mapping={}
T13 10x25->10x25: mapping={}
T14 21x21->10x10: size change (identity)
T15 9x9->9x9: mapping={}
T16 3x3->3x3: mapping={}
T17 21x21->21x21: mapping={}
T18 14x18->14x18: mapping={1: 0, 4: 0}
T19 2x4->4x8: size change (identity)
T20 10x10->10x10: mapping={}
T21 15x15->2x4: size change (identity)
T22 11x11->3x3: size change (identity)
T23 9x11->9x11: mapping={}
T24 9x9->9x9: mapping={0: 2}
T25 18x19->18x19: mapping={}
T26 5x7->5x3: size change (identity)
T27 10x10->10x10: mapping={0: 2}
T28 10x10->10x10: mapping={}
T29 23x21->6x8: size change (identity)
T30 5x10->5x10: mapping={2: 0, 4: 0}
T31 10x12->4x4: size change 

In [3]:
# Build ONNX model for each task
# Conservative: only use mappings consistent across ALL examples
# For size-changing tasks, use identity

models = {}

for t in range(1, 401):
    fname = f'{DATA_DIR}/task{t:03d}.json'
    if not os.path.exists(fname): continue
    with open(fname) as f:
        d = json.load(f)
    
    train = d.get('train', [])
    test = d.get('test', [])
    all_ex = train + test
    
    if not train:
        model = make_1x1({c: c for c in range(CH)})
        tag = 'identity_no_data'
    else:
        ex0 = train[0]
        inp = np.array(ex0['input'])
        out = np.array(ex0['output'])
        IH, IW = inp.shape
        OH, OW = out.shape
        
        if IH != OH or IW != OW:
            model = make_1x1({c: c for c in range(CH)})
            tag = 'identity_size_change'
        else:
            ex_mappings = []
            for ex in all_ex:
                inp_i = np.array(ex['input'])
                out_i = np.array(ex['output'])
                if inp_i.shape != out_i.shape: continue
                m = {}
                for r in range(inp_i.shape[0]):
                    for c in range(inp_i.shape[1]):
                        ic, oc = int(inp_i[r,c]), int(out_i[r,c])
                        if ic != oc:
                            m[ic] = oc
                ex_mappings.append(m)
            
            mapping = {c: c for c in range(CH)}
            
            if ex_mappings:
                for ic in range(CH):
                    oc_vals = [em.get(ic) for em in ex_mappings if ic in em]
                    if oc_vals and len(set(oc_vals)) == 1:
                        oc = oc_vals[0]
                        if ic != oc:
                            mapping[ic] = oc
            
            model = make_1x1(mapping)
            non_id = [k for k, v in mapping.items() if k != v]
            tag = f'1x1_{len(non_id)}_vals' if non_id else 'identity'
    
    mp = f'{OUT_DIR}/task{t:03d}.onnx'
    save(model, mp)
    models[t] = mp
    print(f'Task {t:02d}: {tag}')

Task 01: identity_size_change
Task 02: 1x1_1_vals
Task 03: identity_size_change
Task 04: 1x1_4_vals
Task 05: identity
Task 06: identity_size_change
Task 07: identity
Task 08: 1x1_2_vals
Task 09: identity
Task 10: identity
Task 11: 1x1_2_vals
Task 12: identity
Task 13: identity
Task 14: identity_size_change
Task 15: identity
Task 16: 1x1_8_vals
Task 17: identity
Task 18: 1x1_6_vals
Task 19: identity_size_change
Task 20: identity
Task 21: identity_size_change
Task 22: identity_size_change
Task 23: identity
Task 24: 1x1_1_vals
Task 25: 1x1_5_vals
Task 26: identity_size_change
Task 27: 1x1_1_vals
Task 28: identity
Task 29: identity_size_change
Task 30: 1x1_2_vals
Task 31: identity_size_change
Task 32: 1x1_8_vals
Task 33: identity
Task 34: identity
Task 35: identity
Task 36: identity_size_change
Task 37: identity
Task 38: identity_size_change
Task 39: identity_size_change
Task 40: identity
Task 41: identity
Task 42: 1x1_1_vals
Task 43: 1x1_1_vals
Task 44: 1x1_7_vals
Task 45: identity
Task 4

In [4]:
# Create submission zip
zip_path = f'{OUT_DIR}/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for t in range(1, 401):
        if t in models:
            mp = models[t]
            if os.path.exists(mp):
                zf.write(mp, f'task{t:03d}.onnx')
print(f'Created: {zip_path} ({os.path.getsize(zip_path)} bytes), {len(models)} models')

Created: /kaggle/working/submission.zip (97871 bytes), 400 models
